In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
import zipfile

target_dir = '/content/xmvad'
candidates = [
    '/content/drive/MyDrive/xmvad-ha/xmvad_colab_bundle.zip',
    '/content/xmvad_colab_bundle.zip',
]
src = next((p for p in candidates if os.path.exists(p)), None)
if src is None:
    raise RuntimeError(
        'xmvad_colab_bundle.zip not found. Put it in MyDrive/xmvad-ha/ '
        '(or upload to /content/) then re-run this cell.')
if not os.path.isdir(target_dir):
    os.makedirs(target_dir)
with zipfile.ZipFile(src) as z:
    z.extractall(target_dir)
print('Bundle extracted:', src, '->', target_dir)
print('Contents:', sorted(os.listdir(target_dir)))

In [ ]:
%cd /content/xmvad
import os
assert os.path.isfile('/content/xmvad/requirements_colab.txt'), 'bundle missing requirements_colab.txt'
!pip install -q --no-warn-conflicts -r /content/xmvad/requirements_colab.txt
# bootstrap: mounts Drive paths, locates/extracts the dataset archive from
# MyDrive/xmvad-ha/dataset/, fetches the Point-MAE checkpoint, restores any
# already-computed feature shards from Drive.
!python /content/xmvad/bootstrap.py

In [ ]:
%cd /content/xmvad
import os
os.makedirs('/content/drive/MyDrive/xmvad-ha/logs', exist_ok=True)
# Full pipeline: extract features (if not cached) -> H1/H2/H3 -> comparison.
# Output stays visible in this cell while a copy is tee'd to Drive logs.
!python -u /content/xmvad/run_all.py 2>&1 | tee /content/drive/MyDrive/xmvad-ha/logs/run_all.log

In [ ]:
%cd /content/xmvad
!python /content/xmvad/status.py
print('\nResults on Drive:')
!ls -la /content/drive/MyDrive/xmvad-ha/results || true